# VarLen+CP 长序列设计与正确性验证

本节比较的是两条完整路线：`Varlen+FSDP, seq_len=4096` 与 `Varlen+CP, seq_len=8192`。这里先解释为什么 CP 能在不增加每卡局部序列长度的情况下支持更长数据，以及如何在 TorchTitan-NPU 中启用它；性能异常和跨 rank 归因放到 08.03。

**学习目标**：

- 对齐两条路线的 global token、每卡 local token、QKV shape 和 FSDP mesh；
- 解释 greedy packing 为什么让训练目标近似不变、又为什么不会逐 step 等价；
- 跟踪 `VarlenMetadata`、Ulysses CP 和 NPU TND kernel 的实现链路；
- 在运行前估算 CP AllToAll 的字节量、调用次数和 overlap 风险。

## 1. 先把两条完整路线对齐

`seq_len=8192` 并没有把每张卡的基础工作量翻倍。FSDP+短序列让两张卡各自处理不同的 4096 containers；CP+长序列让两张卡共同处理同一批 8192 containers，并把 sequence/head 所有权互换。

**Varlen+FSDP 短序列**使用 `seq_len=4096, GBS=4, dp_shard=2, cp=1`。一个 step 包含 `4 × 4096 = 16384` 个容器 token；两个 rank 各自取得两个不同的 4096 containers，所以每 rank 的 layer 输入是 8192 token，TND Q shape 为 `[8192,16,128]`。它的 FSDP mesh size 是 `2 × 1 = 2`，没有 Attention AllToAll。

**Varlen+CP 长序列**使用 `seq_len=8192, GBS=2, dp_shard=1, cp=2`。一个 step 同样包含 `2 × 8192 = 16384` 个容器 token，但两个 CP rank 共同处理这两个 8192 containers。sequence 切分后，每 rank 的 layer 输入仍是 `2 × (8192 / 2) = 8192` token；AllToAll 后的 TND Q shape 为 `[16384,8,128]`。它的 FSDP mesh size 同样是 `1 × 2 = 2`，并额外执行 Attention AllToAll。

两种 Q shape 的元素数相同：`8192 × 16 = 16384 × 8`。MLP/Norm 看到的每卡 local token 也同为 8192。因此 CP+8192 的核心不是让每卡做两倍计算，而是把“不同数据、全部 heads”改成“同一份长序列、各一半 heads”。8192 由此提供更长的全局 packing 空间，同时维持与 4096 FSDP 相近的局部 activation 规模；额外代价是 attention 前后的布局通信和临时 buffer。

## 2. Greedy packing 下，为什么训练目标相同或近似

可以把 sequence 看成一个固定容量的箱子，而不是一篇完整文档：

```text
4096 箱子：[样本 A][样本 B][剩余位置]
8192 箱子：[样本 A][样本 B][样本 C][样本 D][剩余位置]
```

Greedy packing 只是把更多独立样本装进更大的箱子。Varlen 保存每条样本的边界，因此 A 只能读取 A，B 只能读取 B；扩大箱子不会把 A、B、C、D 合成一篇长文档。tokenizer、labels、optimizer 和每条样本内部的训练目标都没有改变。

因此，扩大 container 不改变**单条样本内部**的 Attention 与 label 语义，但会改变一个 optimizer step 装入哪些样本、装入多少样本以及 loss 的聚合内容。两条路线不能据此期待逐 step 的 loss 或 gradient 完全相等；比较吞吐时仍要分别统计 raw samples、有效 token 和 supervised token。

## 3. CP 的直接价值：支持更长数据

FSDP 只切分模型参数，不切分一条样本的 sequence。当前 FSDP 配置的 `seq_len=4096`，每张卡都必须独立保存并计算完整的 4096-token 序列，因此它不能承载本章要训练的 8192 长数据。

CP=2 会先把 8192-token sequence 分成两半，每张卡本地只保留 4096 token。进入 Attention 时，AllToAll 把布局从“4096 token × 全部 heads”换成“8192 token × 一半 heads”，计算完成后再换回来：

```text
每 rank 输入：4096 token × 全部 heads
Attention：  8192 token × 一半 heads
每 rank 输出：4096 token × 全部 heads
```

因此 CP+8192 能保留更长的原始样本，也能给 greedy packing 提供更大的箱子，同时每卡 local token 和 Q/K/V 元素数仍与 FSDP+4096 相同。在本章的固定资源和 batch 条件下，CP 是让全局 sequence 从 4096 增至 8192、同时让每卡仍只保留 4096 local token 的路线。08.02 的结论只到这里：**CP 用两张卡共同承载一条更长的 sequence，而不是让每张卡独立承担完整长序列。**

这项能力有通信成本。两条路线的 FSDP mesh 都是 degree 2，参数 all-gather/reduce-scatter 仍然存在；CP+8192 另外增加约 224 次、2.819 GB/rank/microbatch 的 Attention AllToAll，而且 pre-attention AllToAll 是后续计算的前置依赖。它最终是加速还是减速，必须由 08.03 的双 rank trace 判断，不能在本节提前下结论。

In [ ]:
fsdp_local_tokens = 8192
cp_local_tokens = 8192
fsdp_q_elements = 16_777_216
cp_q_elements = 16_777_216

assert fsdp_local_tokens == cp_local_tokens
assert fsdp_q_elements == cp_q_elements
print('FSDP+4096：每 rank 处理 8192 个 local token，Q 有 16,777,216 个元素。')
print('CP+8192：每 rank 也处理 8192 个 local token，Q 元素数相同。')
print('两条路线的 FSDP degree 都是 2。')
print('CP 额外执行 224 次 AllToAll，约 2818.572 MB/rank/microbatch。')


## 4. 实现链路：两个 CP rank 必须使用同一份隔离区间

```text
ChatDataset / Trainer
  → VarlenMetadata(cu_seq_q, cu_seq_k, max_q, max_k)
  → NPUVarlenUlyssesCP pre-hook
       Q/K/V: all_to_all(scatter=head, gather=sequence)
  → NPUVarlenAttention.forward()
       BSND → TND
       cu_seq → CPU int64 actual_seq_qlen / actual_seq_kvlen
       npu_fusion_attention_v3(..., sparse_mode=7)
  → post-hook reverse all_to_all(output)
```

CP pre-hook 后，两个 rank 都看到完整 sequence，但各自只计算一部分 attention heads。因此 `cu_seq_q/cu_seq_k` 必须保持一致，不能让每个 rank 各算一套，否则 token 会被分到错误的隔离区间。当前仓库的实现位置是：`torchtitan_npu/distributed/context_parallel/npu_varlen_cp.py` 负责 pre/post hook，`torchtitan_npu/models/common/npu_varlen_attention.py` 负责 BSND→TND 和 NPU attention kernel。

区间边界由 DataLoader 的位置编号提供：每装入一条新样本，位置编号都会重新从 0 开始；补齐定长 container 时，末尾 padding 也单独从 0 开始。训练器据此生成累计长度；同一样本内部的 `<|im_end|>` 只是消息结束标记，不会把多轮对话拆开。CP pre-hook 必须把这份全局累计长度原样交给两个 rank。

## 5. 配置入口：沿用第 6 章的 TND recipe`sft_qwen3_1_7b_wordle_tnd` 的安装方式与配置细节见 06.05，06.06 已用它完成端到端训练与 profiling，此处不再重复。08.03 会在开跑前再次 import 验证，并写入 `cp=2` 与 `seq_len=16384`；08.04 使用未替换 attention 的基础 `sft_qwen3_1_7b_wordle`。本章的差异不在 recipe 本身，而在并行拓扑参数。

## 练习

1. （单选题）固定 B=2、S=8192、Q heads=16、CP=2 时，Q 在 pre AllToAll 后的 shape 是什么？
    A. [2,4096,16,128]
    B. [2,8192,8,128]
    C. [2,8192,16,128]
    D. [2,4096,8,128]

2. （多选题）VarLen+CP correctness gate 应检查哪些内容？
    A. resolved config 中 cp=2 与正确 inner Attention
    B. 两个 rank 的全局累计长度一致
    C. TND kernel、actual_seq、sparse_mode=7 与有限 loss
    D. 只检查 Python 类名


In [ ]:
!cat ./answer/08.02_answer.txt
